In [1]:
import os
os.chdir("..")

In [4]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from tqdm.notebook import tqdm_notebook
import gc

from mech_interp_toolkit.activation_utils import concat_activations

In [3]:
path = Path("outputs/cached_activations")
original_activations = []
obfuscated_activations = []

max_files = 3

for i, f in enumerate(tqdm_notebook(path.glob("*.pt"))):
    if i > max_files:
        continue
    original, obfuscated = torch.load(f, weights_only=False)
    original.attention_mask = torch.empty(0)
    obfuscated.attention_mask = torch.empty(0)
    original_activations.append(original)
    obfuscated_activations.append(obfuscated)

0it [00:00, ?it/s]

In [ ]:
# Pre-compute activations for all layers
all_layers_original = {}
all_layers_obfuscated = {}

original_activations = concat_activations(original_activations, pad_value=0)
obfuscated_activations = concat_activations(obfuscated_activations, pad_value=0)


for layer in tqdm_notebook(range(28)):
    all_layers_original[layer] = (
        original_activations[(layer, "layer_out")][:, -1, :].float().numpy()
    )
    all_layers_obfuscated[layer] = (
        obfuscated_activations[(layer, "layer_out")][:, -1, :].float().numpy()
    )

del original_activations, obfuscated_activations
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def plot_pca_for_layer(layer):
    """Plot PCA analysis for a specific layer."""
    all_original = all_layers_original[layer]
    all_obfuscated = all_layers_obfuscated[layer]
    
    # Perform PCA on original activations
    pca_original = PCA()
    pca_original.fit(all_original)

    # Get explained variance ratio
    explained_variance_ratio_original = pca_original.explained_variance_ratio_
    cumulative_variance_original = np.cumsum(explained_variance_ratio_original)

    # Find number of components for 90% and 95% variance
    n_components_90_original = np.argmax(cumulative_variance_original >= 0.90) + 1
    n_components_95_original = np.argmax(cumulative_variance_original >= 0.95) + 1

    # Perform PCA on obfuscated activations
    pca_obfuscated = PCA()
    pca_obfuscated.fit(all_obfuscated)

    # Get explained variance ratio
    explained_variance_ratio_obfuscated = pca_obfuscated.explained_variance_ratio_
    cumulative_variance_obfuscated = np.cumsum(explained_variance_ratio_obfuscated)

    # Find number of components for 90% and 95% variance
    n_components_90_obfuscated = np.argmax(cumulative_variance_obfuscated >= 0.90) + 1
    n_components_95_obfuscated = np.argmax(cumulative_variance_obfuscated >= 0.95) + 1

    # Create a single plot with both curves
    plt.figure(figsize=(12, 6))

    # Plot both cumulative variance curves
    plt.plot(range(1, len(cumulative_variance_original) + 1), cumulative_variance_original, 
             linewidth=2, label='Original Activations', color='blue')
    plt.plot(range(1, len(cumulative_variance_obfuscated) + 1), cumulative_variance_obfuscated, 
             linewidth=2, label='Obfuscated Activations', color='orange')

    # Add horizontal reference lines
    plt.axhline(y=0.90, color='red', linestyle='--', linewidth=1.5, label='90% variance', alpha=0.5)
    plt.axhline(y=0.95, color='green', linestyle='--', linewidth=1.5, label='95% variance', alpha=0.5)

    # Add vertical lines for original activations
    plt.axvline(x=n_components_90_original, color='blue', linestyle=':', linewidth=1.5, alpha=0.5)
    plt.axvline(x=n_components_95_original, color='blue', linestyle=':', linewidth=1.5, alpha=0.5)

    # Add vertical lines for obfuscated activations
    plt.axvline(x=n_components_90_obfuscated, color='orange', linestyle=':', linewidth=1.5, alpha=0.5)
    plt.axvline(x=n_components_95_obfuscated, color='orange', linestyle=':', linewidth=1.5, alpha=0.5)

    # Add annotations for original activations
    plt.text(n_components_90_original, 0.88, f'Orig: {n_components_90_original}', 
             ha='center', va='top', fontsize=9, color='blue', 
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='blue', alpha=0.7))
    plt.text(n_components_95_original, 0.85, f'Orig: {n_components_95_original}', 
             ha='center', va='top', fontsize=9, color='blue',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='blue', alpha=0.7))

    # Add annotations for obfuscated activations
    plt.text(n_components_90_obfuscated, 0.92, f'Obf: {n_components_90_obfuscated}', 
             ha='center', va='bottom', fontsize=9, color='orange',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='orange', alpha=0.7))
    plt.text(n_components_95_obfuscated, 0.97, f'Obf: {n_components_95_obfuscated}', 
             ha='center', va='bottom', fontsize=9, color='orange',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='orange', alpha=0.7))

    plt.xlabel('Number of Components', fontsize=12)
    plt.ylabel('Cumulative Variance Explained', fontsize=12)
    plt.title(f'PCA Cumulative Variance: Original vs Obfuscated Activations (Layer {layer})', fontsize=14)
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.xlim(0, max(len(cumulative_variance_original), len(cumulative_variance_obfuscated)))
    plt.ylim(0, 1.0)
    plt.tight_layout()
    plt.show()

    print(f"Layer {layer}:")
    print(f"Original Activations:")
    print(f"  Number of components for 90% variance: {n_components_90_original}")
    print(f"  Number of components for 95% variance: {n_components_95_original}")
    print(f"  Total number of components: {len(cumulative_variance_original)}")

    print(f"\nObfuscated Activations:")
    print(f"  Number of components for 90% variance: {n_components_90_obfuscated}")
    print(f"  Number of components for 95% variance: {n_components_95_obfuscated}")
    print(f"  Total number of components: {len(cumulative_variance_obfuscated)}")

# Create interactive widget
widgets.interact(plot_pca_for_layer, layer=widgets.IntSlider(min=0, max=27, step=1, value=17, description='Layer:'));